# Gibbs & Candès (2021) Adaptive Conformal Inference 复现（SPS-UK）

论文：Isaac Gibbs & Emmanuel J. Candès, **Adaptive Conformal Inference Under Distribution Shift**, NeurIPS 2021.

本 Notebook 使用仓库里已经存在的 **SPS-UK demand**，做一个以“理解 ACI 原理”为目标的方法级复现。它不是原论文股票 / 选举数据上的数值级复现。

我们重点复现论文最核心的四件事：

1. 固定 \(\alpha\) 与自适应 \(\alpha_t\) 的 local coverage 差异；
2. ACI 的长期 empirical miscoverage 是否靠近目标 \(\alpha=0.1\)；
3. 原论文 Proposition 4.1 的长期频率保证是否能在这条序列上数值验证；
4. \(\gamma\) 的 adaptability–stability trade-off，以及 raw / normalized conformity score 的差异。

为了尽量隔离 ACI 本身，这里不训练复杂深度模型，而使用**24 h seasonal persistence** 做 1-step point forecast：

\[
\hat y_t = y_{t-24}.
\]

SPS-UK 原始数据为 30 min，这里重采样为 1 h。这样每个预测时刻都只使用过去已经观测到的信息。


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ALPHA_TARGET = 0.10
GAMMA_PAPER = 0.005
CAL_WINDOW = 1250       # 对齐原论文股票实验的 1250 个近期 score
LOCAL_COV_WINDOW = 500  # 对齐原论文 Figure 1 的局部 coverage 窗口
SEASONAL_LAG = 24       # hourly data: same hour previous day

print("target coverage:", 1 - ALPHA_TARGET)
print("paper gamma:", GAMMA_PAPER)


## 1. 读取与整理 SPS-UK demand

复用本仓库已有数据：

`papers/2024-smartgridcomm-conformal-mlpf/data/SPS-UK_dataset.zip`

这里只使用 demand，不使用 PV / weather。


In [ ]:
def find_sps_zip():
    candidates = [
        Path("../2024-smartgridcomm-conformal-mlpf/data/SPS-UK_dataset.zip"),
        Path("papers/2024-smartgridcomm-conformal-mlpf/data/SPS-UK_dataset.zip"),
        Path("../../papers/2024-smartgridcomm-conformal-mlpf/data/SPS-UK_dataset.zip"),
        Path("data/SPS-UK_dataset.zip"),
    ]
    for p in candidates:
        p = p.resolve()
        if p.exists():
            return p
    roots = [Path.cwd().resolve()] + list(Path.cwd().resolve().parents[:3])
    for root in roots:
        hits = list(root.glob("**/SPS-UK_dataset.zip"))
        if hits:
            return hits[0]
    raise FileNotFoundError("SPS-UK_dataset.zip not found")

def pick_datetime_column(df):
    preferred = [c for c in df.columns if any(k in c.lower() for k in ("datetime","timestamp","date","time"))]
    for c in preferred + list(df.columns):
        parsed = pd.to_datetime(df[c], errors="coerce")
        if parsed.notna().mean() > 0.9:
            return c
    raise ValueError("No datetime column found")

def pick_numeric_column(df):
    nums = list(df.select_dtypes(include=[np.number]).columns)
    named = [c for c in nums if any(k in c.lower() for k in ("demand","power","mw"))]
    if named:
        return named[0]
    if nums:
        return nums[0]
    raise ValueError("No numeric demand column found")

zip_path = find_sps_zip()
print("dataset:", zip_path)

with ZipFile(zip_path) as zf:
    members = {Path(n).name:n for n in zf.namelist()}
    with zf.open(members["demand_train_set4.csv"]) as f:
        raw = pd.read_csv(f)

dt_col = pick_datetime_column(raw)
raw[dt_col] = pd.to_datetime(raw[dt_col], errors="coerce")
raw = raw.dropna(subset=[dt_col]).sort_values(dt_col).drop_duplicates(dt_col)
demand_col = pick_numeric_column(raw)

demand_30m = (
    raw[[dt_col, demand_col]]
    .rename(columns={dt_col:"timestamp", demand_col:"demand_mw"})
    .set_index("timestamp")
    .sort_index()
)

hourly = demand_30m["demand_mw"].resample("1h").mean().dropna().rename("y")
print("30-min:", demand_30m.index.min(), "->", demand_30m.index.max(), "rows=", len(demand_30m))
print("hourly:", hourly.index.min(), "->", hourly.index.max(), "rows=", len(hourly))
display(hourly.head().to_frame())


## 2. 构造在线 1-step 预测与 conformity score

为了避免把“预测器能力”和“ACI 能力”混在一起，这里采用非常透明的 seasonal persistence：

\[
\hat y_t = y_{t-24}.
\]

两种 score：

**Raw absolute residual**

\[
S_t^{raw}=|y_t-\hat y_t|.
\]

**Normalized residual**

\[
S_t^{norm}=\frac{|y_t-\hat y_t|}{\max(|\hat y_t|,\epsilon)}.
\]

normalized score 直接对应原论文 Section 5 的思想：先消除一部分由“量级变化”带来的 score 漂移，再让 ACI 处理剩余 distribution shift。


In [ ]:
df = hourly.to_frame()
df["pred"] = df["y"].shift(SEASONAL_LAG)
df = df.dropna().copy()

# SPS-UK demand 为正；epsilon 只是避免极小预测值造成除法爆炸
EPS_SCALE = max(0.1, 0.05 * float(df["pred"].median()))

df["scale_raw"] = 1.0
df["scale_norm"] = np.maximum(df["pred"].abs(), EPS_SCALE)
df["score_raw"] = (df["y"] - df["pred"]).abs()
df["score_norm"] = df["score_raw"] / df["scale_norm"]

print("usable rows:", len(df))
print("evaluation starts after burn-in/calibration window:", CAL_WINDOW)
print("EPS_SCALE:", round(EPS_SCALE, 4))
display(df.head())


## 3. 复现 ACI 更新

原论文核心更新：

\[
\alpha_{t+1}=\alpha_t+\gamma(\alpha-err_t),
\]

其中：

\[
err_t=\mathbf 1\{Y_t\notin \hat C_t(\alpha_t)\}.
\]

这里**不对 \(\alpha_t\) 做 clip**。这是和原论文理论保持一致的重要细节。

论文在 Proposition 4.1 前规定：

- 当 quantile argument \(p<0\) 时，\(\hat Q_t(p)=-\infty\)；
- 当 \(p>1\) 时，\(\hat Q_t(p)=+\infty\)。

因此若 \(\alpha_t<0\)，下一次 prediction set 自动变为整个实数轴，必然 coverage；若 \(\alpha_t>1\)，prediction set 为空，必然 miscoverage。这个反馈会把 \(\alpha_t\) 推回合理区域，并得到论文 Lemma 4.1 的 \([-\gamma,1+\gamma]\) 有界性。


In [ ]:
def empirical_quantile(scores, p):
    """Paper-style empirical quantile Q(p) with theory-compatible boundaries."""
    if p <= 0:
        return -np.inf
    if p >= 1:
        return np.inf
    s = np.sort(np.asarray(scores, dtype=float))
    k = int(np.ceil(p * len(s))) - 1
    k = min(max(k, 0), len(s) - 1)
    return float(s[k])

def run_online_conformal(data, score_col, scale_col, adaptive, gamma=GAMMA_PAPER,
                         alpha_target=ALPHA_TARGET, cal_window=CAL_WINDOW):
    score_hist = list(data[score_col].iloc[:cal_window].to_numpy(dtype=float))
    alpha_t = float(alpha_target)

    rows = []
    for i in range(cal_window, len(data)):
        yt = float(data["y"].iloc[i])
        pred = float(data["pred"].iloc[i])
        scale = float(data[scale_col].iloc[i])

        alpha_used = alpha_t if adaptive else float(alpha_target)
        q = empirical_quantile(score_hist[-cal_window:], 1 - alpha_used)

        if np.isposinf(q):
            lo, hi = -np.inf, np.inf
        elif np.isneginf(q):
            lo, hi = np.inf, -np.inf
        else:
            radius = q * scale
            lo, hi = pred - radius, pred + radius

        err = int(not (lo <= yt <= hi))
        width = hi - lo if np.isfinite(lo) and np.isfinite(hi) else np.inf

        rows.append({
            "timestamp": data.index[i],
            "y": yt,
            "pred": pred,
            "lower": lo,
            "upper": hi,
            "width": width,
            "alpha_t": alpha_used,
            "q_t": q,
            "err": err,
        })

        # 当前真实值揭示后，score 才能进入后续 calibration window
        score_hist.append(float(data[score_col].iloc[i]))

        if adaptive:
            alpha_t = alpha_t + gamma * (alpha_target - err)

    out = pd.DataFrame(rows).set_index("timestamp")
    out["coverage"] = 1 - out["err"]
    out["local_coverage"] = out["coverage"].rolling(
        LOCAL_COV_WINDOW, min_periods=max(50, LOCAL_COV_WINDOW // 5)
    ).mean()
    out["running_miscoverage"] = out["err"].expanding().mean()
    return out

def summarize_run(name, out):
    finite_width = out.loc[np.isfinite(out["width"]), "width"]
    local = out["local_coverage"].dropna()
    return {
        "method": name,
        "coverage": out["coverage"].mean(),
        "coverage_error": abs(out["coverage"].mean() - (1 - ALPHA_TARGET)),
        "mean_width_finite": finite_width.mean(),
        "infinite_width_rate": 1 - np.isfinite(out["width"]).mean(),
        "local_cov_rmse": np.sqrt(np.mean((local - (1 - ALPHA_TARGET))**2)),
        "alpha_std": out["alpha_t"].std(),
        "alpha_min": out["alpha_t"].min(),
        "alpha_max": out["alpha_t"].max(),
    }


## 4. Fixed \(\alpha\) vs ACI：raw absolute residual

两种方法使用**完全相同的 rolling calibration score window**。唯一差别：

- Fixed：始终使用 \(\alpha_t=0.1\)
- ACI：使用论文更新式动态调整 \(\alpha_t\)

这样可以把性能差异尽量归因于 adaptive alpha 本身。


In [ ]:
fixed_raw = run_online_conformal(df, "score_raw", "scale_raw", adaptive=False)
aci_raw = run_online_conformal(df, "score_raw", "scale_raw", adaptive=True, gamma=GAMMA_PAPER)

main_metrics = pd.DataFrame([
    summarize_run("Fixed alpha / raw score", fixed_raw),
    summarize_run("ACI gamma=0.005 / raw score", aci_raw),
])

display(main_metrics.round(5))


In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(fixed_raw.index, fixed_raw["local_coverage"], label="Fixed alpha")
ax.plot(aci_raw.index, aci_raw["local_coverage"], label="ACI")
ax.axhline(1 - ALPHA_TARGET, linestyle="--", label="Target coverage = 0.90")
ax.set_title(f"Local coverage over most recent {LOCAL_COV_WINDOW} predictions")
ax.set_ylabel("Coverage")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(14, 3.5))
ax.plot(aci_raw.index, aci_raw["alpha_t"])
ax.axhline(ALPHA_TARGET, linestyle="--", label="Target alpha = 0.10")
ax.set_title("Adaptive alpha trajectory")
ax.set_ylabel("alpha_t")
ax.legend()
plt.show()


## 5. 数值检查 Proposition 4.1

原论文 Proposition 4.1 给出：

\[
\left|
\frac{1}{T}\sum_{t=1}^T err_t-\alpha
\right|
\le
\frac{\max\{\alpha_1,1-\alpha_1\}+\gamma}{T\gamma}.
\]

它的重点不是这个 bound 很紧，而是：

\[
\frac{1}{T}\sum_{t=1}^T err_t \to \alpha.
\]

下面直接用本次 ACI 的 error sequence 检查左侧是否落在该理论上界内。


In [ ]:
T = np.arange(1, len(aci_raw) + 1)
running_err = np.cumsum(aci_raw["err"].to_numpy()) / T
gap = np.abs(running_err - ALPHA_TARGET)
bound = (max(ALPHA_TARGET, 1 - ALPHA_TARGET) + GAMMA_PAPER) / (T * GAMMA_PAPER)

prop_check = bool(np.all(gap <= bound + 1e-12))
print("Proposition 4.1 numerical bound respected:", prop_check)
print("final empirical miscoverage:", round(running_err[-1], 6))
print("final |miscoverage-alpha|:", round(gap[-1], 6))
print("final theoretical bound:", round(bound[-1], 6))

fig, ax = plt.subplots(figsize=(14, 4))
start = 100
ax.plot(T[start:], gap[start:], label="|running miscoverage - alpha|")
ax.plot(T[start:], bound[start:], linestyle="--", label="Proposition 4.1 bound")
ax.set_yscale("log")
ax.set_xlabel("T")
ax.set_ylabel("Absolute gap")
ax.set_title("Numerical check of Proposition 4.1")
ax.legend()
plt.show()


## 6. Score 设计：raw residual vs normalized residual

原论文 Section 5 强调：ACI 的效果高度依赖 conformity score 是否尽可能接近 stationary。

这里不把“stationary”当成一个二元标签，而做一个简单诊断：把 evaluation period 按时间分成 6 段，比较每段 score 的均值与 90% quantile。如果某种 score 的这些统计量跨时间更稳定，只能说明它在这个数据上的**尺度漂移更小**，不能据此证明严格平稳。


In [ ]:
eval_scores = df.iloc[CAL_WINDOW:].copy()
n_blocks = 6
block_id = np.floor(np.arange(len(eval_scores)) * n_blocks / len(eval_scores)).astype(int)
block_id = np.clip(block_id, 0, n_blocks - 1)
eval_scores["block"] = block_id + 1

block_stats = (
    eval_scores.groupby("block")
    .agg(
        raw_mean=("score_raw", "mean"),
        raw_q90=("score_raw", lambda x: np.quantile(x, 0.90)),
        norm_mean=("score_norm", "mean"),
        norm_q90=("score_norm", lambda x: np.quantile(x, 0.90)),
    )
)

raw_q90_cv = block_stats["raw_q90"].std() / block_stats["raw_q90"].mean()
norm_q90_cv = block_stats["norm_q90"].std() / block_stats["norm_q90"].mean()

display(block_stats.round(4))
print("Across-block CV of score q90")
print("  raw :", round(raw_q90_cv, 4))
print("  norm:", round(norm_q90_cv, 4))


In [ ]:
fixed_norm = run_online_conformal(df, "score_norm", "scale_norm", adaptive=False)
aci_norm = run_online_conformal(df, "score_norm", "scale_norm", adaptive=True, gamma=GAMMA_PAPER)

score_metrics = pd.DataFrame([
    summarize_run("Fixed / raw", fixed_raw),
    summarize_run("ACI / raw", aci_raw),
    summarize_run("Fixed / normalized", fixed_norm),
    summarize_run("ACI / normalized", aci_norm),
])
display(score_metrics.round(5))

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(aci_raw.index, aci_raw["local_coverage"], label="ACI / raw")
ax.plot(aci_norm.index, aci_norm["local_coverage"], label="ACI / normalized")
ax.axhline(1 - ALPHA_TARGET, linestyle="--", label="Target = 0.90")
ax.set_title("ACI local coverage: raw vs normalized score")
ax.set_ylabel("Coverage")
ax.legend()
plt.show()


## 7. \(\gamma\) sensitivity：adaptability vs stability

原论文 Section 2.1 的核心判断：

- \(\gamma\) 大：适应 distribution shift 更快，但 \(\alpha_t\) 更容易波动；
- \(\gamma\) 小：轨迹更稳定，但可能跟不上 shift；
- 原论文真实数据实验采用 \(\gamma=0.005\)。

这里比较 \(0.001,0.005,0.02\)。


In [ ]:
gammas = [0.001, 0.005, 0.02]
gamma_runs = {
    g: run_online_conformal(df, "score_raw", "scale_raw", adaptive=True, gamma=g)
    for g in gammas
}

gamma_metrics = pd.DataFrame([
    summarize_run(f"ACI gamma={g}", gamma_runs[g]) | {"gamma": g}
    for g in gammas
]).sort_values("gamma")
display(gamma_metrics.round(5))

fig, ax = plt.subplots(figsize=(14, 4))
for g in gammas:
    ax.plot(gamma_runs[g].index, gamma_runs[g]["alpha_t"], label=f"gamma={g}")
ax.axhline(ALPHA_TARGET, linestyle="--", label="Target alpha")
ax.set_title("Step-size sensitivity: alpha_t trajectories")
ax.set_ylabel("alpha_t")
ax.legend()
plt.show()


## 8. 本次复现应该怎样解读

这次复现只回答“ACI 机制是否按论文描述工作”，不试图复现原论文股票和选举实验的数值。

应重点检查：

1. ACI 的长期 empirical miscoverage 是否靠近 \(\alpha=0.1\)；
2. Proposition 4.1 的数值上界是否成立；
3. Fixed 与 ACI 的 local coverage 波动是否存在差异；
4. normalized score 在 SPS-UK 上是否真的比 raw score 更稳定——这件事**不预设答案**；
5. \(\gamma\) 增大时，\(\alpha_t\) 波动是否变强。

特别注意：原论文理论设置是假定 \(Y_t\) 每一步及时揭示。本 Notebook 也是即时反馈的 one-step setting；它**没有**验证 48-step multi-horizon 下 delayed feedback 的理论保证。


In [ ]:
print("=== Reproduction summary ===")
print(f"Fixed/raw coverage: {fixed_raw['coverage'].mean():.4f}")
print(f"ACI/raw coverage:   {aci_raw['coverage'].mean():.4f}")
print(f"ACI/raw miscoverage:{aci_raw['err'].mean():.4f}  (target={ALPHA_TARGET:.4f})")
print(f"Proposition 4.1 bound check: {prop_check}")
print(f"raw q90 block-CV:   {raw_q90_cv:.4f}")
print(f"norm q90 block-CV:  {norm_q90_cv:.4f}")
print()
if norm_q90_cv < raw_q90_cv:
    print("On this SPS-UK run, normalized score has a more stable block-wise q90 than raw score.")
else:
    print("On this SPS-UK run, normalized score does NOT improve block-wise q90 stability over raw score.")
